# AI Curriculum Analysis with JAAT

This notebook applies the [JAAT (Job Ad Analysis Toolkit)](https://github.com/Job-Ad-Research-at-QSB-LUC/JAAT) to AI course syllabi to extract:
1. **Tasks** (O*NET standardized work tasks)
2. **Skills** (ESCO/EuropaCodes skills)
3. **AI Concepts** (Specific AI-related tasks and requirements)

## 1. Setup

**Per-course reporting includes a _strict_ AI score** — the average of matched AI-statement scores, restricted to statements with a curated weight greater than 0.


In [ ]:
!pip install -q JAAT pandas openpyxl matplotlib seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 65.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 66.1 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
from JAAT import JAAT
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
nltk.download('punkt_tab', quiet=True)

# Setup for Google Colab
file_path = 'AI_course_syllabi_.xlsx'
if not os.path.exists(file_path):
    print("File not found locally. Attempting to clone from GitHub...")
    !git clone --depth 1 https://github.com/pnorlander/JAAT_Demos.git
    if os.path.exists('JAAT_Demos/ai_curriculum/AI_course_syllabi_.xlsx'):
        os.chdir('JAAT_Demos/ai_curriculum')
        print(f"Switched to {os.getcwd()}")
    else:
        print("Error: Could not find the dataset in the cloned repository.")

sns.set_theme(style="whitegrid")

# Load the data
if os.path.exists(file_path):
    df = pd.read_excel(file_path)
    print(f"Loaded {len(df)} courses.")
else:
    print(f"Warning: {file_path} not found. Please ensure it is in the same directory.")

File not found locally. Attempting to clone from GitHub...
Cloning into 'JAAT_Demos'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 18 (delta 1), reused 9 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 4.96 MiB | 15.12 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Switched to /content/JAAT_Demos/Curriculum
Loaded 113 courses.


## 2. Preprocessing

We combine the course description, content/topics, and outcomes into a single text block for analysis.

In [ ]:
# Fill NaNs with empty strings
text_cols = [
    'What is the course description ?',
    'What is the course  content/topics?',
    'What are the course outcomes/objectives?'
]
for col in text_cols:
    df[col] = df[col].fillna('')

# Combine into a single text column
df['combined_text'] = df[text_cols].apply(lambda x: '\n'.join(x), axis=1)

print("Sample combined text:")
if len(df) > 0:
    print(df['combined_text'].iloc[0][:500] + "...")

Sample combined text:
CS 4365 - Artificial Intelligence (3 semester credit hours) Basic concepts and techniques that enable
computers to perform intelligent tasks. Examples are taken from areas such as natural language
understanding, computer vision, machine learning, search strategies and control, logic, and theorem
proving. Prerequisite: CE 3345 or CS 3345 or SE 3345 or TE 3345 or equivalent. (3-0) Y
not mentioned
1. Understand and use uninformed and heuristic search techniques
2. Understand and use local search al...


## 3. Initialize JAAT Modules

In [ ]:
tm = JAAT.TaskMatch()
sm = JAAT.SkillMatch()
ai = JAAT.AIMatch()

INIT
Preparing embeddings...


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

Finished.


## 4. Run Analysis

We will run the analysis on each course. Note: For large datasets, use `get_tasks_batch`, `get_skills_batch`, and `get_ai_batch` for better performance.

In [ ]:
print("Starting JAAT analysis... this may take a few minutes.")

# 1. Split into sentences for better TaskMatch/SkillMatch performance
print("Splitting text into sentences...")
sentence_data = []
for idx, row in df.iterrows():
    text = row["combined_text"]
    for s in nltk.sent_tokenize(text):
        if len(s.strip()) > 5:
            sentence_data.append({"course_idx": idx, "sentence": s})

sent_df = pd.DataFrame(sentence_data)
print(f"Created {len(sent_df)} sentences from {len(df)} courses.")

# 2. Run TaskMatch and SkillMatch on sentences
print("Running TaskMatch and SkillMatch on sentences...")
sent_list = sent_df["sentence"].tolist()
sent_df["tasks"]  = tm.get_tasks_batch(sent_list)
sent_df["skills"] = sm.get_skills_batch(sent_list)

# 3. Aggregate back to course level (dedup by code to avoid double-counting identical matches)
print("Aggregating matches back to course level...")
def _dedup_by_code(matches_list, code_index):
    seen, out = set(), []
    for sub in matches_list:
        for item in sub:
            code = item[code_index]
            if code not in seen:
                seen.add(code)
                out.append(tuple(item))
    return out

# TaskMatch returns (Task ID, Task label) -> code at index 0
course_tasks  = sent_df.groupby("course_idx")["tasks"].apply(
    lambda lst: _dedup_by_code(lst, code_index=0))
# SkillMatch returns (Skill label, Code) -> code at index 1
course_skills = sent_df.groupby("course_idx")["skills"].apply(
    lambda lst: _dedup_by_code(lst, code_index=1))

df["tasks"]  = df.index.map(course_tasks).apply(lambda x: x if isinstance(x, list) else [])
df["skills"] = df.index.map(course_skills).apply(lambda x: x if isinstance(x, list) else [])

# 4. Run AIMatch on full text (it handles broader context better)
# get_ai returns: (matched_ai[(Statement, Code)], count, avg_score_all, binary_scores, match_scores)
print("Running AIMatch on full course text...")
ai_results = df["combined_text"].apply(lambda x: ai.get_ai(x) or ([], 0, 0, [], []))
df["ai_concepts"]      = ai_results.apply(lambda x: x[0])
df["ai_concept_count"] = ai_results.apply(lambda x: x[1])

# 5. Three per-course AI scores
#
# For every matched AI statement we look up its curated Score in ai.score_map
# (the "Score" column from JAAT's ai_a6_5_redacted_final2.csv). We then
# summarize those Scores three ways:
#
#   ai_avg_score     AVERAGE: JAAT default -- mean of Score across ALL matched
#                    AI statements (Score=0 matches drag the mean down).
#                    == get_ai()[2]
#
#   ai_strict_score  STRICT : mean of Score restricted to matched statements
#                    with Score > 0. Excludes zero-weighted matches entirely.
#
#   ai_lenient_score LENIENT: SUM of Score across all matched statements.
#                    Rewards breadth -- a syllabus with many weakly-weighted
#                    matches can still score highly.
def _scores_for(concepts):
    return [ai.score_map.get(stmt, 0) for stmt, _code in concepts]

def ai_avg_score(concepts):
    scores = _scores_for(concepts)
    return round(sum(scores) / len(scores), 3) if scores else 0.0

def ai_strict_score(concepts):
    pos = [s for s in _scores_for(concepts) if s > 0]
    return round(sum(pos) / len(pos), 3) if pos else 0.0

def ai_lenient_score(concepts):
    scores = _scores_for(concepts)
    return round(sum(scores), 3) if scores else 0.0

def strict_ai_count(concepts):
    return sum(1 for s in _scores_for(concepts) if s > 0)

df["ai_avg_score"]     = df["ai_concepts"].apply(ai_avg_score)
df["strict_ai_score"]  = df["ai_concepts"].apply(ai_strict_score)
df["lenient_ai_score"] = df["ai_concepts"].apply(ai_lenient_score)
df["strict_ai_count"]  = df["ai_concepts"].apply(strict_ai_count)

print("Analysis complete.")
print(f"  Courses with >=1 matched AI concept:        {(df['ai_concept_count'] > 0).sum()}")
print(f"  Courses with >=1 strict (Score>0) concept:  {(df['strict_ai_count']  > 0).sum()}")
print(f"  Mean AVG score  (courses with >=1 match):   "
      f"{df.loc[df['ai_concept_count']>0, 'ai_avg_score'].mean():.3f}")
print(f"  Mean STRICT score (courses with strict>0):  "
      f"{df.loc[df['strict_ai_score']>0, 'strict_ai_score'].mean():.3f}")
print(f"  Mean LENIENT score (courses with >=1 match):"
      f"{df.loc[df['ai_concept_count']>0, 'lenient_ai_score'].mean():.3f}")


## 5. Explore Results

In [ ]:
title_col = "What is the course title ? "

if len(df) > 0:
    row0 = df.iloc[0]
    print(f"Course: {row0[title_col]}")
    print(f"Tasks (first 5):  {[label for _code, label in row0['tasks'][:5]]}")
    print(f"Skills (first 5): {[label for label, _code in row0['skills'][:5]]}")
    print(f"AI Concepts (first 5): {[stmt for stmt, _code in row0['ai_concepts'][:5]]}")
    print(f"AI concept count:            {row0['ai_concept_count']}")
    print(f"Strict AI count (Score>0):   {row0['strict_ai_count']}")
    print(f"AI Avg score (all matched):  {row0['ai_avg_score']:.3f}")
    print(f"AI Strict score (Score>0):   {row0['strict_ai_score']:.3f}")
    print(f"AI Lenient score (sum):      {row0['lenient_ai_score']:.3f}")

# Per-course leaderboard: top 10 by strict AI score
print("\n--- Top 10 courses by strict AI score ---")
print(df[[title_col, "strict_ai_count",
         "ai_avg_score", "strict_ai_score", "lenient_ai_score"]]
      .sort_values("strict_ai_score", ascending=False)
      .head(10)
      .to_string(index=False))


## 6. Visualizations

We lead with **AIMatch** results. Three per-course AI scores:

- **Average** (`ai_avg_score`): mean of curated Scores across *all* matched AI statements. JAAT default.
- **Strict** (`strict_ai_score`): mean of Scores restricted to matched statements with Score > 0.
- **Lenient** (`lenient_ai_score`): *sum* of Scores across all matched statements — rewards breadth.

Every chart has axis labels and data labels so it reads standalone.


In [ ]:
from collections import Counter

sns.set_theme(style="whitegrid")
title_col = "What is the course title ? "

def _wrap(s, width=55):
    s = str(s)
    return s if len(s) <= width else s[:width - 1] + "\u2026"

def _flatten_labels(series, label_index):
    """Flatten list-of-tuples into human-readable labels.
    label_index=0 for (label, code); label_index=1 for (code, label)."""
    out = []
    for sub in series:
        for item in sub:
            if isinstance(item, (list, tuple)) and len(item) > label_index:
                out.append(str(item[label_index]))
            else:
                out.append(str(item))
    return out

def _bar_with_labels(counts, title, xlabel, ylabel, color="#2E86AB",
                     figsize=(11, 6), value_fmt="{:d}"):
    if not counts:
        print(f"No data for: {title}")
        return
    plot_df = pd.DataFrame(counts, columns=[ylabel, xlabel])
    plot_df[ylabel] = plot_df[ylabel].apply(_wrap)
    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(data=plot_df, y=ylabel, x=xlabel, color=color, ax=ax)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    for p in ax.patches:
        w = p.get_width()
        ax.annotate(value_fmt.format(w),
                    (w, p.get_y() + p.get_height() / 2),
                    ha="left", va="center", xytext=(4, 0),
                    textcoords="offset points", fontsize=10)
    ax.margins(x=0.08)
    plt.tight_layout(); plt.show()

# ---------------------------------------------------------------
# 1. LEAD: Compare Average vs. Strict vs. Lenient per course
#    (side-by-side bars for the top 10 courses, ranked by strict)
# ---------------------------------------------------------------
top = (df[[title_col, "strict_ai_count",
           "ai_avg_score", "strict_ai_score", "lenient_ai_score"]]
       .sort_values("strict_ai_score", ascending=False)
       .head(10).copy())
top["label"] = top[title_col].astype(str).apply(_wrap, width=45)

plot_df = top.melt(
    id_vars=["label"],
    value_vars=["ai_avg_score", "strict_ai_score", "lenient_ai_score"],
    var_name="Score type", value_name="Score",
)
label_map = {
    "ai_avg_score":     "Average",
    "strict_ai_score":  "Strict",
    "lenient_ai_score": "Lenient (sum)",
}
plot_df["Score type"] = plot_df["Score type"].map(label_map)

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(data=plot_df, y="label", x="Score", hue="Score type",
            palette={"Average": "#2E86AB",
                     "Strict":  "#C73E1D",
                     "Lenient (sum)": "#6A994E"}, ax=ax)
ax.set_title("Top 10 Courses by Strict AI Score — Average vs. Strict vs. Lenient",
             fontsize=13, fontweight="bold")
ax.set_xlabel("AI Score")
ax.set_ylabel("Course")
for p in ax.patches:
    w = p.get_width()
    if w and w > 0:
        ax.annotate(f"{w:.2f}",
                    (w, p.get_y() + p.get_height() / 2),
                    ha="left", va="center", xytext=(3, 0),
                    textcoords="offset points", fontsize=8)
ax.legend(title="Score type", loc="lower right")
ax.margins(x=0.15)
plt.tight_layout(); plt.show()

# ---------------------------------------------------------------
# 2. Distribution of STRICT AI scores across syllabi
# ---------------------------------------------------------------
strict = df.loc[df["strict_ai_score"] > 0, "strict_ai_score"]
fig, ax = plt.subplots(figsize=(11, 6))
sns.histplot(strict, bins=20, kde=True, color="#C73E1D", ax=ax)
mean_s, median_s = strict.mean(), strict.median()
ax.axvline(mean_s,   color="black", linestyle="--", linewidth=1.2,
           label=f"Mean = {mean_s:.3f}")
ax.axvline(median_s, color="gray",  linestyle=":",  linewidth=1.2,
           label=f"Median = {median_s:.3f}")
ax.set_title(
    f"Distribution of Strict AI Scores "
    f"(n={len(strict)} of {len(df)} courses)",
    fontsize=13, fontweight="bold")
ax.set_xlabel("Strict AI Score (avg of matched statement Scores, >0 only)")
ax.set_ylabel("Number of courses")
for p in ax.patches:
    h = p.get_height()
    if h > 0:
        ax.annotate(f"{int(h)}",
                    (p.get_x() + p.get_width() / 2, h),
                    ha="center", va="bottom",
                    xytext=(0, 2), textcoords="offset points", fontsize=9)
ax.legend()
plt.tight_layout(); plt.show()

# ---------------------------------------------------------------
# 3. Strict score vs. Lenient score
#    Separates "few high-Score matches" from "many weak matches"
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 6))
mask = df["ai_concept_count"] > 0
ax.scatter(df.loc[mask, "lenient_ai_score"],
           df.loc[mask, "strict_ai_score"],
           s=40, alpha=0.6, color="#C73E1D", edgecolor="white")
ax.set_title("Strict vs. Lenient AI Score per Syllabus",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Lenient AI Score (sum of Scores across all matches)")
ax.set_ylabel("Strict AI Score (avg of Score>0 matches)")
for _, row in df.nlargest(5, "strict_ai_score").iterrows():
    ax.annotate(_wrap(row[title_col], width=40),
                (row["lenient_ai_score"], row["strict_ai_score"]),
                xytext=(5, 3), textcoords="offset points", fontsize=8)
plt.tight_layout(); plt.show()

# ---------------------------------------------------------------
# 4. Top 10 AI concepts (statements) extracted by AIMatch
# ---------------------------------------------------------------
ai_counts = Counter(_flatten_labels(df["ai_concepts"], label_index=0)).most_common(10)
_bar_with_labels(
    ai_counts,
    title="Top 10 AI Concepts Across AI Syllabi (AIMatch)",
    xlabel="Number of courses mentioning concept",
    ylabel="AI Concept (statement)",
    color="#C73E1D",
)

# ---------------------------------------------------------------
# 5. Supporting: Top O*NET Tasks (TaskMatch) - label is at index 1
# ---------------------------------------------------------------
task_counts = Counter(_flatten_labels(df["tasks"], label_index=1)).most_common(10)
_bar_with_labels(
    task_counts,
    title="Top 10 O*NET Tasks Across AI Syllabi (TaskMatch)",
    xlabel="Number of courses mentioning task",
    ylabel="O*NET Task",
    color="#2E86AB",
)

# ---------------------------------------------------------------
# 6. Supporting: Top ESCO Skills (SkillMatch) - label is at index 0
# ---------------------------------------------------------------
skill_counts = Counter(_flatten_labels(df["skills"], label_index=0)).most_common(10)
_bar_with_labels(
    skill_counts,
    title="Top 10 ESCO Skills Across AI Syllabi (SkillMatch)",
    xlabel="Number of courses mentioning skill",
    ylabel="ESCO Skill",
    color="#6A994E",
)


## 7. Save Results

In [ ]:
output_file = 'AI_course_syllabi_analyzed.xlsx'
df.to_excel(output_file, index=False)
print(f"Results saved to {output_file}")